# Appendix A — Legacy Prototype Event Study

**Status:** Superseded by the refined Section 7 baseline.

**Purpose:** Historical comparison, regression verification, and research traceability.

**Authority:** Not authoritative for future strategy decisions. The code, saved outputs, figures, and original conclusions below are preserved as legacy evidence and must not be counted as independent current evidence.

## A.1 Legacy Research Contract

Research contract:

```text
1. Treat each candidate row as an event-study observation, not as a trade in a portfolio.
2. Preserve uncapped forward R, MFE_R, and MAE_R so runner tails remain visible.
3. Use capped 5R/10R diagnostics and robust quantiles when ranking groups, so a few rare large wins do not dominate the research.
4. Respect execution rules: no new candidate entries after 12:00pm New York time; positions may continue after noon; all event-study holding windows are capped at the same-day 3:30pm New York forced-exit boundary.
5. Report first-touch views separately because one POI can produce many retests and one retest can produce multiple candidate variants.
```

T-stats are diagnostic only. Candidate rows are not fully independent because many rows share POIs, retests, market days, and overlapping forward paths.

## A.2 Legacy Inputs and Validation

The reusable implementation lives in `src/features/poi_event_study.py`. The generated Section 7 parquet outputs are reproducible artifacts and remain excluded from Git by `.gitignore`.

In [ ]:
import time
from pathlib import Path

project_root_path = Path(project_root)
if str(project_root_path) not in sys.path:
    sys.path.insert(0, str(project_root_path))

from src.features.poi_event_study import (
    SECTION7_BAR_REQUIRED_COLUMNS,
    SECTION7_SIGNAL_REQUIRED_COLUMNS,
    Section7Config,
    build_section7_event_study,
    build_section7_validation,
    save_section7_outputs,
)

section7_config = Section7Config()
section7_output_dir = project_root_path / "data" / "processed"
section7_signal_path = section7_output_dir / "section6b_structural_signal_frame_gc.parquet"
section7_research_bars_path = section7_output_dir / "research_bars_gc_mgc_1m.parquet"
section7_summary_path = section7_output_dir / "section7_event_study_summary_gc.parquet"
section7_ranking_path = section7_output_dir / "section7_candidate_signal_ranking_gc.parquet"

section7_signal_frame = pd.read_parquet(section7_signal_path)
section7_research_bars = pd.read_parquet(
    section7_research_bars_path,
    columns=SECTION7_BAR_REQUIRED_COLUMNS,
)

section7_validation = build_section7_validation(
    signal_frame=section7_signal_frame,
    research_bars=section7_research_bars,
    config=section7_config,
)

section7_validation

signal_row_count                            1597373
research_bar_row_count                      3487656
signal_required_columns_present                True
bar_required_columns_present                   True
missing_signal_columns                             
missing_bar_columns                                
signal_ids_unique                              True
existing_5_15_30_60_forward_r_present          True
structural_columns_present                     True
structural_partition_valid                     True
session_columns_present                        True
entry_rows_in_approved_execution_windows       True
no_candidate_entry_after_1200_ny               True
risk_points_positive                           True
valid_candidate_flag_all_true                  True
volatility_context_present                     True
volume_context_present                         True
gc_research_bars_available                     True
dtype: object

In [ ]:
section7_start_time = time.perf_counter()

if section7_summary_path.exists() and section7_ranking_path.exists():
    section7_event_study_summary = pd.read_parquet(section7_summary_path)
    section7_candidate_signal_ranking = pd.read_parquet(section7_ranking_path)
    section7_runtime_seconds = 0.0
    section7_output_paths = {
        "event_study_summary": section7_summary_path,
        "candidate_signal_ranking": section7_ranking_path,
    }
else:
    section7_outputs = build_section7_event_study(
        signal_frame=section7_signal_frame,
        research_bars=section7_research_bars,
        config=section7_config,
    )
    section7_output_paths = save_section7_outputs(
        outputs=section7_outputs,
        output_dir=section7_output_dir,
    )
    section7_event_study_summary = section7_outputs["section7_event_study_summary"]
    section7_candidate_signal_ranking = section7_outputs["section7_candidate_signal_ranking"]
    section7_runtime_seconds = time.perf_counter() - section7_start_time

pd.Series(
    {
        "summary_rows": len(section7_event_study_summary),
        "ranking_rows": len(section7_candidate_signal_ranking),
        "runtime_seconds_if_rebuilt": section7_runtime_seconds,
        **{name: str(path) for name, path in section7_output_paths.items()},
    }
)

summary_rows                                                             176560
ranking_rows                                                                374
runtime_seconds_if_rebuilt                                                  0.0
event_study_summary           C:\Users\abond\Desktop\WORK FILES\Systemic\Pro...
candidate_signal_ranking      C:\Users\abond\Desktop\WORK FILES\Systemic\Pro...
dtype: object

## A.3 Legacy Baseline Forward-R Study

Both fixed-horizon and forced-exit-capped labels are reported. Fixed 240m and 360m horizons lose observations when the requested horizon extends beyond 3:30pm New York time. Capped labels evaluate late-morning entries through the forced-exit boundary instead of pretending they can hold past it.

In [ ]:
baseline_cols = [
    "horizon_minutes",
    "metric_mode",
    "valid_count",
    "mean_r",
    "mean_r_cap_5",
    "median_r",
    "q10_r",
    "q25_r",
    "q75_r",
    "q90_r",
    "positive_r_rate",
    "hit_pos_1r_rate",
    "hit_pos_2r_rate",
    "hit_pos_3r_rate",
    "hit_pos_5r_rate",
    "hit_pos_10r_rate",
    "hit_neg_1r_rate",
    "mean_mfe_r",
    "mean_mfe_r_cap_5",
    "mean_mae_r",
    "forced_exit_rate",
]

baseline_forward_r_summary = (
    section7_event_study_summary
    .query("group_spec == 'baseline_all_candidates'")
    .sort_values(["metric_mode", "horizon_minutes"])
    .loc[:, baseline_cols]
)

baseline_forward_r_summary

,horizon_minutes,metric_mode,valid_count,mean_r,mean_r_cap_5,median_r,q10_r,q25_r,q75_r,q90_r,...,hit_pos_1r_rate,hit_pos_2r_rate,hit_pos_3r_rate,hit_pos_5r_rate,hit_pos_10r_rate,hit_neg_1r_rate,mean_mfe_r,mean_mfe_r_cap_5,mean_mae_r,forced_exit_rate
11035,5,capped,1597238,-0.000397,0.001240,0.000000,-1.476190,-0.648148,0.656250,1.484848,...,0.370370,0.131327,0.054936,0.013940,0.001426,0.366548,1.063301,1.033064,1.062291,0.000000
33105,15,capped,1596853,-0.009295,0.003019,0.000000,-2.444444,-1.045455,1.065574,2.452055,...,0.554288,0.284535,0.154483,0.054694,0.007190,0.550309,1.723858,1.578241,1.739226,0.000000
55175,30,capped,1596406,0.000822,0.011020,0.000000,-3.400000,-1.471698,1.493151,3.463415,...,0.663587,0.410406,0.258219,0.114129,0.020524,0.660284,2.404484,2.038089,2.423347,0.000000
77245,60,capped,1594815,-0.016987,0.008856,0.000000,-4.846154,-2.022989,2.062500,4.812500,...,0.752060,0.537869,0.384514,0.207016,0.055443,0.749705,3.410577,2.544956,3.443676,0.000000
99315,120,capped,1590235,-0.013151,0.011959,0.000000,-6.758621,-2.741935,2.787879,6.680000,...,0.816740,0.645514,0.505250,0.316831,0.113583,0.815264,4.708706,3.016889,4.736921,0.000000
121385,180,capped,1583476,-0.022164,0.030028,0.039216,-8.170732,-3.259259,3.333333,7.954545,...,0.845936,0.697248,0.569651,0.382224,0.155189,0.844907,5.626928,3.263981,5.683066,0.000000
143455,240,capped,1576988,-0.056391,0.024891,0.050000,-9.380952,-3.785714,3.794872,9.180000,...,0.861773,0.727555,0.612032,0.430332,0.191104,0.861057,6.340773,3.424738,6.411540,0.083994
165525,360,capped,1578933,-0.079348,0.007673,0.000000,-11.268293,-4.592593,4.536232,11.000000,...,0.876299,0.755423,0.648916,0.476689,0.236422,0.876867,7.298989,3.571832,7.414124,0.467175
0,5,fixed,1597238,-0.000397,0.001240,0.000000,-1.476190,-0.648148,0.656250,1.484848,...,0.370370,0.131327,0.054936,0.013940,0.001426,0.366548,1.063301,1.033064,1.062291,0.000000
22070,15,fixed,1596853,-0.009295,0.003019,0.000000,-2.444444,-1.045455,1.065574,2.452055,...,0.554288,0.284535,0.154483,0.054694,0.007190,0.550309,1.723858,1.578241,1.739226,0.000000


Baseline interpretation:

```text
The all-candidate POI stream is not a clean edge by itself.
Across horizons, positive-R rates sit near 50%, medians are near zero, and uncapped means are mixed to negative.
Runner potential is real: +5R hit rates rise from roughly 1.4% at 5m to roughly 47.7% at capped 360m.
The cost is large adverse path risk: -1R path-hit rates rise from roughly 36.7% at 5m to roughly 87.7% at capped 360m.
```

This supports the Section 7 premise: raw POI logic is a baseline event generator, not a finished trading signal.

## A.4 Legacy POI Touch Event Study

First-touch results are separated from the all-retouch candidate pool to reduce duplicate-weighting risk. Candidate rows are still variant-level observations because entry and stop definitions change the R outcome.

In [ ]:
touch_specs = [
    "primary_first_touch_candidates",
    "poi_touch_direction",
    "poi_touch_order",
    "poi_touch_number",
    "poi_age_minutes",
    "poi_age_candles",
]

touch_event_summary_60m = (
    section7_event_study_summary
    .query("group_spec in @touch_specs and metric_mode == 'capped' and horizon_minutes == 60")
    .sort_values(["group_spec", "mean_r_cap_5"], ascending=[True, False])
)

touch_event_summary_60m[
    [
        "group_spec",
        "group_key",
        "valid_count",
        "mean_r_cap_5",
        "median_r",
        "positive_r_rate",
        "hit_pos_3r_rate",
        "hit_pos_5r_rate",
        "hit_neg_1r_rate",
    ]
].head(40)

,group_spec,group_key,valid_count,mean_r_cap_5,median_r,positive_r_rate,hit_pos_3r_rate,hit_pos_5r_rate,hit_neg_1r_rate
77285,poi_age_candles,candles_since_poi_bucket=120-240|direction=bea...,150111,0.288618,0.184211,0.525271,0.441440,0.276389,0.726496
77277,poi_age_candles,candles_since_poi_bucket=15-30|direction=bearish,74669,0.100048,0.038462,0.504212,0.355409,0.186932,0.734281
77282,poi_age_candles,candles_since_poi_bucket=<=15|direction=bearish,110185,0.057144,-0.080000,0.484948,0.366593,0.189109,0.734156
77283,poi_age_candles,candles_since_poi_bucket=30-60|direction=bearish,105014,0.050801,-0.034483,0.493096,0.358962,0.192193,0.721723
77284,poi_age_candles,candles_since_poi_bucket=60-120|direction=bearish,143654,0.038318,-0.081967,0.485750,0.355375,0.196688,0.703635
77286,poi_age_candles,candles_since_poi_bucket=60-120|direction=bullish,154062,0.031616,0.087977,0.511638,0.343719,0.166277,0.722664
77288,poi_age_candles,candles_since_poi_bucket=>240|direction=bearish,191569,0.010652,-0.029412,0.495874,0.500624,0.314957,0.788144
77281,poi_age_candles,candles_since_poi_bucket=>240|direction=bullish,206014,-0.000875,0.076923,0.506393,0.441140,0.238756,0.812823
77280,poi_age_candles,candles_since_poi_bucket=30-60|direction=bullish,111108,-0.043789,0.038462,0.504626,0.325773,0.147811,0.736752
77279,poi_age_candles,candles_since_poi_bucket=15-30|direction=bullish,76019,-0.073020,0.000000,0.498441,0.305055,0.124140,0.733251


## A.5 Legacy Structural Validation Study

This comparison asks whether higher-order 15/21/31-bar structural swing validation improves the forward event distribution versus local-only 3/5/7-bar swing breaks.

In [ ]:
structural_specs = ["structural_validation", "structural_window", "structural_partition"]

structural_event_summary = (
    section7_event_study_summary
    .query("group_spec in @structural_specs and metric_mode == 'capped'")
    .sort_values(["group_spec", "group_key", "horizon_minutes"])
)

structural_event_summary[
    [
        "group_spec",
        "horizon_minutes",
        "group_key",
        "valid_count",
        "mean_r_cap_5",
        "median_r",
        "positive_r_rate",
        "hit_pos_3r_rate",
        "hit_pos_5r_rate",
        "hit_neg_1r_rate",
    ]
].head(80)

,group_spec,horizon_minutes,group_key,valid_count,mean_r_cap_5,median_r,positive_r_rate,hit_pos_3r_rate,hit_pos_5r_rate,hit_neg_1r_rate
11086,structural_partition,5,structural_swing_break_flag=False|local_swing_...,911915,-0.000589,0.000000,0.496952,0.057596,0.014759,0.381070
33156,structural_partition,15,structural_swing_break_flag=False|local_swing_...,911766,-0.002204,0.000000,0.498359,0.160892,0.057948,0.563855
55226,structural_partition,30,structural_swing_break_flag=False|local_swing_...,911586,0.005740,0.010638,0.500126,0.266746,0.119367,0.670509
77296,structural_partition,60,structural_swing_break_flag=False|local_swing_...,910752,0.005121,0.018182,0.500587,0.395995,0.213846,0.758290
99366,structural_partition,120,structural_swing_break_flag=False|local_swing_...,907990,0.016214,0.000000,0.499615,0.518381,0.326418,0.823285
...,...,...,...,...,...,...,...,...,...,...
77293,structural_window,60,structural_window_bucket=none,910752,0.005121,0.018182,0.500587,0.395995,0.213846,0.758290
99363,structural_window,120,structural_window_bucket=none,907990,0.016214,0.000000,0.499615,0.518381,0.326418,0.823285
121433,structural_window,180,structural_window_bucket=none,904258,0.016998,0.040000,0.502714,0.583318,0.392210,0.852248
143503,structural_window,240,structural_window_bucket=none,900461,0.026683,0.081081,0.505428,0.625705,0.440383,0.866587


Structural interpretation:

```text
Structural validation does not uniformly dominate local-only POIs.
The 15-bar structural cohort is the cleanest structural split at 60m, with positive capped mean and median R.
The 21-bar and 31-bar cohorts are weaker at 60m, although the broader structurally validated group regains some capped strength at longer horizons.
```

Structural validation should move forward as a candidate filter/interactor, not as a blanket permission rule.

## A.6 Legacy Swing Window and Break-Mode Study

In [ ]:
swing_break_summary_60m = (
    section7_event_study_summary
    .query("group_spec == 'swing_break_mode' and metric_mode == 'capped' and horizon_minutes == 60")
    .sort_values("mean_r_cap_5", ascending=False)
)

swing_break_summary_60m[
    [
        "group_key",
        "valid_count",
        "mean_r",
        "mean_r_cap_5",
        "median_r",
        "positive_r_rate",
        "hit_pos_3r_rate",
        "hit_pos_5r_rate",
        "hit_neg_1r_rate",
    ]
]

,group_key,valid_count,mean_r,mean_r_cap_5,median_r,positive_r_rate,hit_pos_3r_rate,hit_pos_5r_rate,hit_neg_1r_rate
77298,swing_n=5|break_mode=wick|direction=bearish,144716,0.323712,0.131570,0.040000,0.504464,0.413354,0.242226,0.734825
77297,swing_n=5|break_mode=close|direction=bearish,117573,0.267086,0.111531,0.021739,0.500838,0.411974,0.243168,0.737261
77302,swing_n=7|break_mode=wick|direction=bearish,151702,0.269634,0.088126,0.000000,0.496922,0.406066,0.233273,0.735567
77304,swing_n=3|break_mode=close|direction=bearish,103770,0.233697,0.077924,0.000000,0.499778,0.417018,0.243587,0.742498
77303,swing_n=3|break_mode=wick|direction=bearish,132364,0.217836,0.077785,0.014709,0.500408,0.413602,0.240103,0.740798
77301,swing_n=7|break_mode=close|direction=bearish,125077,0.210465,0.048509,-0.069767,0.488635,0.399514,0.231601,0.742878
77300,swing_n=7|break_mode=wick|direction=bullish,155586,-0.180591,-0.012657,0.047947,0.505174,0.363349,0.180601,0.752870
77299,swing_n=5|break_mode=wick|direction=bullish,158349,-0.260578,-0.050433,0.013333,0.500193,0.363993,0.180096,0.762632
77305,swing_n=3|break_mode=wick|direction=bullish,143705,-0.271106,-0.063055,0.000000,0.498299,0.358985,0.176438,0.757051
77307,swing_n=5|break_mode=close|direction=bullish,126372,-0.294100,-0.088895,-0.034483,0.493851,0.362691,0.176843,0.767306


Swing/break interpretation:

```text
The 60m study shows a strong side asymmetry: bearish/short POI variants rank above bullish/long variants across swing and break modes.
5-bar wick bearish and 5-bar close bearish groups are among the stronger broad slices, but downside risk remains high.
Do not select swing_n or break_mode only by mean R; sample size, median, q10, and stop-hit behavior still matter.
```

## A.7 Legacy Entry Variant and Stop-Model Study

In [ ]:
entry_stop_summary = (
    section7_event_study_summary
    .query("group_spec == 'entry_stop_model' and metric_mode == 'capped'")
    .sort_values(["group_key", "horizon_minutes"])
)

entry_stop_summary[
    [
        "horizon_minutes",
        "group_key",
        "valid_count",
        "mean_r_cap_5",
        "median_r",
        "positive_r_rate",
        "hit_pos_3r_rate",
        "hit_pos_5r_rate",
        "hit_neg_1r_rate",
    ]
].head(100)

,horizon_minutes,group_key,valid_count,mean_r_cap_5,median_r,positive_r_rate,hit_pos_3r_rate,hit_pos_5r_rate,hit_neg_1r_rate
11125,5,entry_variant=boundary|stop_model=conservative...,359493,-0.005722,0.000000,0.497545,0.026078,0.004612,0.296276
33195,15,entry_variant=boundary|stop_model=conservative...,359400,-0.024317,0.020000,0.501338,0.087451,0.022602,0.486185
55265,30,entry_variant=boundary|stop_model=conservative...,359280,-0.026623,0.022727,0.501522,0.170544,0.056482,0.609285
77335,60,entry_variant=boundary|stop_model=conservative...,358860,-0.057510,0.000000,0.497737,0.286669,0.121518,0.715599
99405,120,entry_variant=boundary|stop_model=conservative...,357638,-0.100489,-0.048780,0.491860,0.407328,0.212603,0.788806
...,...,...,...,...,...,...,...,...,...
77340,60,entry_variant=midpoint|stop_model=poi_distal_e...,45619,0.093629,0.033708,0.501480,0.598786,0.408251,0.853372
99410,120,entry_variant=midpoint|stop_model=poi_distal_e...,45601,0.232380,0.178571,0.513717,0.707090,0.544703,0.891998
121480,180,entry_variant=midpoint|stop_model=poi_distal_e...,45553,0.298390,0.255814,0.515312,0.751015,0.600268,0.905209
143550,240,entry_variant=midpoint|stop_model=poi_distal_e...,45436,0.326239,0.415929,0.525376,0.788648,0.647020,0.913945


Entry/stop interpretation:

```text
Short boundary entries are the cleanest broad entry family.
Short boundary + POI distal edge shows the strongest 60m capped behavior among large entry/stop groups.
Midpoint and distal entries can increase runner hit rates, but they also push -1R path-hit rates higher.
Long variants are generally weak or negative after robust capping, especially midpoint/distal variants.
```

## A.8 Legacy Session-Specific Results

In [ ]:
session_summary = (
    section7_event_study_summary
    .query("group_spec == 'execution_session' and metric_mode == 'capped'")
    .sort_values(["group_key", "horizon_minutes"])
)

session_summary[
    [
        "horizon_minutes",
        "group_key",
        "valid_count",
        "mean_r_cap_5",
        "median_r",
        "positive_r_rate",
        "hit_pos_3r_rate",
        "hit_pos_5r_rate",
        "hit_neg_1r_rate",
    ]
]

,horizon_minutes,group_key,valid_count,mean_r_cap_5,median_r,positive_r_rate,hit_pos_3r_rate,hit_pos_5r_rate,hit_neg_1r_rate
11159,5,execution_window_label=London Execution|trade_...,170080,-0.019719,0.000000,0.496349,0.019444,0.002952,0.291657
33229,15,execution_window_label=London Execution|trade_...,169969,-0.046947,0.000000,0.499009,0.077038,0.016556,0.478381
55299,30,execution_window_label=London Execution|trade_...,169834,-0.080416,-0.031250,0.489843,0.153685,0.039444,0.604084
77369,60,execution_window_label=London Execution|trade_...,169021,-0.183043,-0.111111,0.477118,0.263411,0.093763,0.713876
99439,120,execution_window_label=London Execution|trade_...,167465,-0.238919,-0.161290,0.474840,0.384218,0.171618,0.805846
121509,180,execution_window_label=London Execution|trade_...,166736,-0.111057,-0.041667,0.493733,0.486644,0.260166,0.851142
143579,240,execution_window_label=London Execution|trade_...,166160,0.015246,0.192982,0.516027,0.578719,0.382204,0.879032
165649,360,execution_window_label=London Execution|trade_...,166160,-0.039434,0.052632,0.502437,0.686988,0.515816,0.925361
11161,5,execution_window_label=London Execution|trade_...,161080,0.017424,0.000000,0.492513,0.031500,0.005916,0.283815
33231,15,execution_window_label=London Execution|trade_...,160836,0.034569,0.000000,0.492166,0.110578,0.037380,0.477990


Session interpretation:

```text
London short candidates show the best 60m broad-session profile.
New York short candidates improve most at 120-360m capped horizons, consistent with slower continuation/runner behavior.
London long candidates are weak and should be deprioritized.
```

## A.9 Legacy Volatility-Regime Results

In [ ]:
volatility_summary = (
    section7_event_study_summary
    .query("group_spec == 'volatility_regime' and metric_mode == 'capped'")
    .sort_values(["group_key", "horizon_minutes"])
)

volatility_summary[
    [
        "horizon_minutes",
        "group_key",
        "valid_count",
        "mean_r_cap_5",
        "median_r",
        "positive_r_rate",
        "hit_pos_3r_rate",
        "hit_pos_5r_rate",
        "hit_neg_1r_rate",
    ]
].head(120)

,horizon_minutes,group_key,valid_count,mean_r_cap_5,median_r,positive_r_rate,hit_pos_3r_rate,hit_pos_5r_rate,hit_neg_1r_rate
11181,5,retest_volatility_regime=elevated|trade_side=long,119772,0.013696,0.000000,0.496251,0.010303,0.001962,0.208045
33251,15,retest_volatility_regime=elevated|trade_side=long,119653,0.024700,0.032258,0.506180,0.054817,0.010739,0.405431
55321,30,retest_volatility_regime=elevated|trade_side=long,119504,0.064168,0.032258,0.504962,0.133761,0.038049,0.550115
77391,60,retest_volatility_regime=elevated|trade_side=long,119223,0.131310,0.076923,0.512762,0.260629,0.101264,0.670273
99461,120,retest_volatility_regime=elevated|trade_side=long,118431,0.187952,0.219512,0.529422,0.410450,0.207978,0.746511
...,...,...,...,...,...,...,...,...,...
77399,60,retest_volatility_regime=unknown|trade_side=short,263,-0.021055,-0.697674,0.319392,0.494297,0.091255,0.885932
99469,120,retest_volatility_regime=unknown|trade_side=short,263,-0.912920,-1.761905,0.311787,0.528517,0.091255,1.000000
121539,180,retest_volatility_regime=unknown|trade_side=short,263,-1.997007,-2.371429,0.182510,0.528517,0.273764,1.000000
143609,240,retest_volatility_regime=unknown|trade_side=short,263,-3.244697,-4.371429,0.136882,0.528517,0.273764,1.000000


Volatility interpretation:

```text
Volatility is a real interaction variable, not a simple on/off edge filter.
Extreme-volatility shorts are consistently stronger than extreme-volatility longs.
Elevated-volatility longs are one of the few broad long-side slices that remain positive after capping, especially at longer horizons.
Low/unknown volatility buckets have much smaller samples and should not be over-weighted.
```

## A.10 Legacy Volume and Relative-Volume Study

In [ ]:
volume_specs = ["relative_volume", "volume_zscore", "volume_spike"]

volume_summary_60m = (
    section7_event_study_summary
    .query("group_spec in @volume_specs and metric_mode == 'capped' and horizon_minutes == 60")
    .sort_values(["group_spec", "mean_r_cap_5"], ascending=[True, False])
)

volume_summary_60m[
    [
        "group_spec",
        "group_key",
        "valid_count",
        "mean_r_cap_5",
        "median_r",
        "positive_r_rate",
        "hit_pos_3r_rate",
        "hit_pos_5r_rate",
        "hit_neg_1r_rate",
    ]
].head(80)

,group_spec,group_key,valid_count,mean_r_cap_5,median_r,positive_r_rate,hit_pos_3r_rate,hit_pos_5r_rate,hit_neg_1r_rate
77410,relative_volume,relative_volume_bucket=unknown|trade_side=long,121,0.787910,1.312500,0.743802,0.264463,0.082645,0.520661
77400,relative_volume,relative_volume_bucket=normal|trade_side=short,165524,0.163588,0.111111,0.513472,0.431134,0.255570,0.748157
77408,relative_volume,relative_volume_bucket=high|trade_side=short,77263,0.163309,0.121212,0.513493,0.450811,0.274815,0.760286
77401,relative_volume,relative_volume_bucket=low|trade_side=short,309382,0.082571,-0.024096,0.495475,0.399519,0.229810,0.729457
77407,relative_volume,relative_volume_bucket=spike|trade_side=short,88545,0.034292,-0.040816,0.493975,0.470100,0.287345,0.786911
77409,relative_volume,relative_volume_bucket=very_low|trade_side=short,134265,0.012737,-0.093750,0.482032,0.345094,0.186005,0.703832
77404,relative_volume,relative_volume_bucket=very_low|trade_side=long,139286,-0.034366,0.031250,0.502721,0.298429,0.132892,0.711543
77403,relative_volume,relative_volume_bucket=spike|trade_side=long,100673,-0.036348,0.038462,0.503650,0.423659,0.223953,0.807118
77406,relative_volume,relative_volume_bucket=high|trade_side=long,82566,-0.064468,-0.010101,0.497566,0.401436,0.206320,0.794564
77402,relative_volume,relative_volume_bucket=low|trade_side=long,320219,-0.072928,0.000000,0.498518,0.348280,0.166605,0.748753


Volume interpretation:

```text
Relative volume helps separate movement magnitude, especially on shorts, but spike flags alone also increase adverse path risk.
Normal/high relative-volume shorts are cleaner at 60m than spike-only shorts.
Volume should be tested as a sizing/target/hold-time interaction, not as a standalone permission filter yet.
```

## A.11 Legacy Trend, VWAP, Extension, and Exhaustion Study

In [ ]:
context_specs = [
    "trend_alignment",
    "vwap_alignment",
    "vwap_extension",
    "directional_efficiency",
    "poi_quality",
]

context_summary_60m = (
    section7_event_study_summary
    .query("group_spec in @context_specs and metric_mode == 'capped' and horizon_minutes == 60")
    .sort_values(["group_spec", "mean_r_cap_5"], ascending=[True, False])
)

context_summary_60m[
    [
        "group_spec",
        "group_key",
        "valid_count",
        "mean_r_cap_5",
        "median_r",
        "positive_r_rate",
        "hit_pos_3r_rate",
        "hit_pos_5r_rate",
        "hit_neg_1r_rate",
    ]
].head(100)

,group_spec,group_key,valid_count,mean_r_cap_5,median_r,positive_r_rate,hit_pos_3r_rate,hit_pos_5r_rate,hit_neg_1r_rate
77456,directional_efficiency,directional_efficiency_bucket=efficient|trade_...,1291,0.833583,0.800000,0.627421,0.496514,0.230829,0.810999
77458,directional_efficiency,directional_efficiency_bucket=unknown|trade_si...,136,0.568533,1.185185,0.705882,0.250000,0.088235,0.558824
77453,directional_efficiency,directional_efficiency_bucket=directional|trad...,30289,0.138315,0.206897,0.524976,0.425006,0.229621,0.776222
77454,directional_efficiency,directional_efficiency_bucket=choppy|trade_sid...,547582,0.110376,0.031250,0.502191,0.409208,0.234710,0.737785
77450,directional_efficiency,directional_efficiency_bucket=mixed|trade_side...,196619,0.075726,-0.032787,0.493965,0.408923,0.247819,0.738057
...,...,...,...,...,...,...,...,...,...
77499,poi_quality,poi_size_bucket=medium|fvg_size_bucket=5-10|br...,5410,-0.520195,-0.535714,0.439556,0.376895,0.191497,0.808503
77536,poi_quality,poi_size_bucket=very_large|fvg_size_bucket=5-1...,3588,-0.651427,-0.926471,0.391583,0.444816,0.261148,0.848105
77532,poi_quality,poi_size_bucket=small|fvg_size_bucket=2-5|brea...,73,-0.691037,-0.285714,0.369863,0.493151,0.219178,0.657534
77518,poi_quality,poi_size_bucket=large|fvg_size_bucket=1-2|brea...,11587,-0.726113,-0.943662,0.401657,0.404591,0.239406,0.825926


Context interpretation:

```text
Short candidates near or moderately extended from VWAP are cleaner than very-extended short candidates at 60m.
Long candidates against VWAP are weak and should be rejected for first backtest scope.
Trend alignment is not a universal positive filter; short candidates against the 20/50 trend bias performed better than aligned shorts in the broad 60m slice.
POI-quality buckets contain some extreme small-sample winners, so they are useful for hypothesis generation but should be ignored unless they pass minimum sample thresholds.
```

## A.12 Legacy Candidate Signal Ranking

Ranking uses the capped forward-R diagnostics, minimum sample thresholds, downside quantiles, MFE/MAE behavior, runner hit rates, and consistency across horizons. It intentionally does not rank solely by mean R.

In [ ]:
ranking_cols = [
    "rank",
    "rank_score",
    "valid_horizon_count",
    "min_valid_count",
    "mean_r_cap_5_avg",
    "median_r_avg",
    "q10_r_avg",
    "positive_r_rate_avg",
    "hit_pos_3r_rate_avg",
    "hit_pos_5r_rate_avg",
    "hit_neg_1r_rate_avg",
    "group_key",
]

section7_candidate_signal_ranking[ranking_cols].head(30)

,rank,rank_score,valid_horizon_count,min_valid_count,mean_r_cap_5_avg,median_r_avg,q10_r_avg,positive_r_rate_avg,hit_pos_3r_rate_avg,hit_pos_5r_rate_avg,hit_neg_1r_rate_avg,group_key
0,1,-0.261748,7,1004,0.355790,0.294234,-3.599541,0.536598,0.423921,0.245257,0.642218,structural_validation_bucket=structural_valida...
1,2,-0.266627,8,1102,0.341385,0.384853,-4.551437,0.546081,0.361840,0.223443,0.624572,structural_validation_bucket=local_swing_only|...
2,3,-0.272807,8,1257,0.392213,0.326227,-3.723994,0.535381,0.380081,0.247406,0.651089,structural_validation_bucket=structural_valida...
3,4,-0.276794,8,1172,0.339119,0.298173,-3.353301,0.523580,0.374402,0.256061,0.627850,structural_validation_bucket=structural_valida...
4,5,-0.278668,8,1183,0.407391,0.419491,-5.111705,0.547640,0.487306,0.338799,0.692953,structural_validation_bucket=structural_valida...
5,6,-0.283198,8,1125,0.045636,0.209128,-4.889555,0.528725,0.323971,0.144961,0.648022,structural_validation_bucket=structural_valida...
6,7,-0.283888,8,1111,0.371783,0.364275,-4.421435,0.545738,0.445692,0.287531,0.668701,structural_validation_bucket=structural_valida...
7,8,-0.289279,8,1524,0.394087,0.462994,-5.193421,0.539802,0.463607,0.317218,0.694229,structural_validation_bucket=local_swing_only|...
8,9,-0.290512,8,1356,0.406796,0.452125,-4.951762,0.547446,0.478808,0.335281,0.696864,structural_validation_bucket=structural_valida...
9,10,-0.292491,8,1135,0.129818,0.240893,-4.593291,0.537706,0.307586,0.164291,0.572035,structural_validation_bucket=local_swing_only|...


: 

Ranking interpretation:

```text
No composite group earns a clean robust approval.
The highest-ranked groups still have negative robust scores because downside quantiles and -1R path-hit rates are severe.
Several groups do have positive capped mean/median R and strong runner rates; these are Section 8 research-backtest candidates, not final trade rules.
```

## A.13 Legacy Summary and Historical Backtest Candidates

> Legacy conclusion retained for traceability; superseded by the authoritative Section 7 baseline.

Section 7 is complete.

Rejected or deprioritized for first Section 8 scope:

```text
1. The raw all-candidate POI stream as a standalone signal.
2. The assumption that first touch is automatically cleaner.
3. London long candidates.
4. Long candidates against VWAP or in extreme-volatility broad slices.
5. Volume-spike-only filters as a permission rule.
6. Tiny POI-quality buckets with attractive means but insufficient sample size.
```

Promising candidates for Section 8 first research backtest:

```text
1. Short-only POI candidates, especially New York extreme-volatility shorts with boundary entries and POI-distal or conservative stops.
2. London short 60m reaction candidates, especially boundary-entry variants.
3. New York short 120-240m runner candidates with forced-exit-aware holding logic.
4. The 15-bar structural validation cohort as an interaction/filter, not a blanket rule.
5. Elevated-volatility long candidates as the only broad long-side family worth a limited test.
```

Research conclusion:

```text
The current POI framework behaves less like a universal continuation edge and more like a conditional event generator with asymmetric short-side opportunity and large two-sided excursion.
Longer holding windows increase runner potential, but they also increase -1R path-hit risk.
Section 8 should start narrow, forced-exit-aware, and short-side-first rather than backtesting every raw POI candidate.
```

Saved Section 7 outputs:

```text
data/processed/section7_event_study_summary_gc.parquet
data/processed/section7_candidate_signal_ranking_gc.parquet
```

## A.14 Legacy Visual Audit Utility

This subsection is for algorithm validation only, not performance evaluation. It randomly samples candidate POI events and overlays the detected POI zone, entry, stop, targets, and relevant timestamps on 1-minute GC research bars.

Visual audit batches:

```text
1. London short boundary candidates
2. New York short boundary candidates
3. Elevated-volatility long candidates, when sample size is sufficient
```

Saved chart directory:

```text
reports/figures/section7_visual_audit/
```

In [ ]:
from src.features.poi_event_study import (
    SECTION7_VISUAL_AUDIT_BAR_COLUMNS,
    SECTION7_VISUAL_AUDIT_SIGNAL_COLUMNS,
    build_section7_visual_audit_batches,
)

section7_visual_audit_dir = project_root_path / "reports" / "figures" / "section7_visual_audit"
section7_visual_audit_manifest_path = section7_visual_audit_dir / "section7_visual_audit_manifest.csv"

section7_visual_signal_frame = pd.read_parquet(
    section7_signal_path,
    columns=SECTION7_VISUAL_AUDIT_SIGNAL_COLUMNS,
)
section7_visual_research_bars = pd.read_parquet(
    section7_research_bars_path,
    columns=SECTION7_VISUAL_AUDIT_BAR_COLUMNS,
)

section7_visual_audit_manifest = build_section7_visual_audit_batches(
    signal_frame=section7_visual_signal_frame,
    research_bars=section7_visual_research_bars,
    output_dir=section7_visual_audit_dir,
    samples_per_batch=12,
    random_state=714,
    min_sample_size=12,
    context_minutes_before=90,
    context_minutes_after=180,
    forward_r_column="forward_60m_r",
)
section7_visual_audit_manifest.to_csv(section7_visual_audit_manifest_path, index=False)

section7_visual_audit_manifest.groupby(["batch_name", "status"], observed=True).size().rename("chart_count")

In [ ]:
section7_visual_audit_manifest[
    [
        "batch_name",
        "status",
        "available_count",
        "sample_position",
        "signal_id",
        "chart_path",
    ]
].head(20)

In [ ]:
from IPython.display import Image, display

inline_visual_audit_paths = (
    section7_visual_audit_manifest
    .loc[section7_visual_audit_manifest["status"].eq("plotted"), "chart_path"]
    .head(10)
    .to_list()
)

for chart_path in inline_visual_audit_paths:
    path = Path(chart_path)
    if not path.is_absolute():
        path = project_root_path / path
    display(Image(filename=str(path)))

Inline visual audit samples from the generated batch:

![london_short_boundary_01_GC_SIG_00253578](../../reports/figures/section7_visual_audit/london_short_boundary/london_short_boundary_01_GC_SIG_00253578.png)

![london_short_boundary_02_GC_SIG_00275503](../../reports/figures/section7_visual_audit/london_short_boundary/london_short_boundary_02_GC_SIG_00275503.png)

![london_short_boundary_03_GC_SIG_00280932](../../reports/figures/section7_visual_audit/london_short_boundary/london_short_boundary_03_GC_SIG_00280932.png)

![london_short_boundary_04_GC_SIG_00398552](../../reports/figures/section7_visual_audit/london_short_boundary/london_short_boundary_04_GC_SIG_00398552.png)

![london_short_boundary_05_GC_SIG_00478780](../../reports/figures/section7_visual_audit/london_short_boundary/london_short_boundary_05_GC_SIG_00478780.png)

![london_short_boundary_06_GC_SIG_00669827](../../reports/figures/section7_visual_audit/london_short_boundary/london_short_boundary_06_GC_SIG_00669827.png)

![london_short_boundary_07_GC_SIG_00743835](../../reports/figures/section7_visual_audit/london_short_boundary/london_short_boundary_07_GC_SIG_00743835.png)

![london_short_boundary_08_GC_SIG_00773397](../../reports/figures/section7_visual_audit/london_short_boundary/london_short_boundary_08_GC_SIG_00773397.png)

![london_short_boundary_09_GC_SIG_00902302](../../reports/figures/section7_visual_audit/london_short_boundary/london_short_boundary_09_GC_SIG_00902302.png)

![london_short_boundary_10_GC_SIG_00967259](../../reports/figures/section7_visual_audit/london_short_boundary/london_short_boundary_10_GC_SIG_00967259.png)



Visual audit interpretation:

```text
These plots are sanity checks for the POI/candidate engine.
They help verify whether the shaded POI zone, retest/entry timestamp, entry price, stop, targets, structural classification, and forced-exit boundary line up with the underlying 1-minute GC bars.
They should not be used to estimate performance or choose parameters by eye.
```